GABUNGKAN SELURUH DATA YANG SUDAH DISIMPAN

In [2]:
import pandas as pd

# =====================================================
# 1. LOAD DATASET
# =====================================================
df1 = pd.read_csv('data_jumlah_puskesmas.csv')
df2 = pd.read_csv('data_jumlah_tenagakesehatan.csv')
df3 = pd.read_csv('data_jumlah_penyakittidakmenular.csv')
df4 = pd.read_csv('data_jumlah_penyakitmenular.csv')
df5 = pd.read_csv('data_jumlah_penduduk.csv')

datasets = [df1, df2, df3, df4, df5]

# =====================================================
# 2. STANDARISASI NAMA KOLOM
# =====================================================
for df in datasets:
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )

# =====================================================
# 3. STANDARISASI WILAYAH & TAHUN
# =====================================================
for df in datasets:

    df['nama_kabupaten_kota'] = (
        df['nama_kabupaten_kota']
        .str.upper()
        .str.strip()
    )

    df['tahun'] = (
        df['tahun']
        .astype(str)
        .str.extract(r'(\d{4})')[0]
        .astype(int)
    )

# =====================================================
# 4. MERGE 5 DATASET
# =====================================================
df_merged = datasets[0]

for i in range(1, len(datasets)):
    df_merged = df_merged.merge(
        datasets[i],
        on=[
            'kode_kabupaten_kota',
            'nama_kabupaten_kota',
            'tahun'
        ],
        how='outer'
    )

# =====================================================
# 5. FILTER TAHUN
# =====================================================
df_merged = df_merged[
    df_merged['tahun'].between(2018, 2024)
]

# =====================================================
# 6. PERBAIKI TIPE DATA KODE
# =====================================================
for col in ['kode_provinsi', 'kode_kabupaten_kota']:
    df_merged[col] = (
        df_merged[col]
        .fillna(0)
        .astype(int)
        .replace(0, pd.NA)
    )

# =====================================================
# 7. SORT DATA
# =====================================================
df_merged = df_merged.sort_values(
    ['tahun', 'kode_kabupaten_kota']
).reset_index(drop=True)

# =====================================================
# 8. CEK DUPLIKASI
# =====================================================
print(
    "Jumlah duplikasi:",
    df_merged.duplicated(
        subset=['kode_kabupaten_kota', 'tahun']
    ).sum()
)

df_merged = df_merged.drop(columns=['unnamed:_0'], errors='ignore')


df_merged


Jumlah duplikasi: 0


,kode_provinsi,nama_provinsi,kode_kabupaten_kota,nama_kabupaten_kota,tahun,pelayanan_non_rawat_inap,pelayanan_rawat_inap,jumlah_bidan,jumlah_dokter,jumlah_perawat,jumlah_penyakit_tidak_menular,jumlah_penyakit_menular,jumlah_penduduk
0,35,JAWA TIMUR,3501,KABUPATEN PACITAN,2018,9,15,347,141,619,41173,292,592938
1,35,JAWA TIMUR,3502,KABUPATEN PONOROGO,2018,12,19,519,129,990,46778,479,962125
2,35,JAWA TIMUR,3503,KABUPATEN TRENGGALEK,2018,3,19,384,227,792,40464,285,748432
3,35,JAWA TIMUR,3504,KABUPATEN TULUNGAGUNG,2018,14,18,609,263,1098,114929,507,1109547
4,35,JAWA TIMUR,3505,KABUPATEN BLITAR,2018,6,18,540,233,837,65232,392,1230159
...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,35,JAWA TIMUR,3575,KOTA PASURUAN,2024,8,0,1,0,261,18844,287,213469
262,35,JAWA TIMUR,3576,KOTA MOJOKERTO,2024,6,0,4,0,396,16010,129,142272
263,35,JAWA TIMUR,3577,KOTA MADIUN,2024,6,0,5,0,521,23764,756,201733
264,35,JAWA TIMUR,3578,KOTA SURABAYA,2024,40,23,69,0,7446,851331,2299,3018022


In [3]:
# =====================================================
# 9. SIMPAN
# =====================================================
df_merged.to_csv(
    "dataset_fasilitas_kesehatan_skripsi_kedua.csv",
    index=False,
    float_format="%.0f"
)